# Correlations

1. Edit `curve_config.CORR_SERIES` / scatter legs
2. Run cells below → heatmaps + `exports/correlations/corr_panel.xlsx`

| knobs | default |
|---|---|
| `CORR_WINDOW` | 50 bars (~rolling corr) |
| `CORR_PCT_LOOKBACK` | 200 (percentile history) |
| scatter | ρ(USGG5YR, CL1) vs ρ(USGG5YR, ESA) |

Missing tickers auto-pull via bdib; CO1/CL1 fall back to daily bdh if `NOT_ENTITLED`.

In [ ]:
import importlib, logging, os, sys
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "lib" / "correlations.py").exists() else _DZ
if not (REPO / "lib" / "correlations.py").exists():
    raise FileNotFoundError("No lib/correlations.py — git pull origin main")
os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())

In [ ]:
import run_dz
importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
import curve_config
import lib.correlations as correlations
import lib.data as data
importlib.reload(curve_config)
importlib.reload(data)
importlib.reload(correlations)
print("CORR_SERIES:", curve_config.CORR_SERIES)
print("window=%s  pct_lookback=%s" % (
    curve_config.CORR_WINDOW, curve_config.CORR_PCT_LOOKBACK))

In [ ]:
level, pmat, full = correlations.run_correlations()
print("asof:", full.index[-1], "n=", len(full))
print("export:", correlations.LAST_EXPORT_PATH)
display(level.round(3))
display(pmat.round(1))

In [ ]:
g = Path(curve_config.GRAPH_DIR_CORR)
for name in ("corr_level_heatmap.png", "corr_percentile_heatmap.png",
            "corr_scatter_rates_assets.png"):
    p = g / name
    display(Image(filename=str(p))) if p.exists() else print("missing", name)
if correlations.LAST_SCATTER_DF is not None:
    display(correlations.LAST_SCATTER_DF.tail(8))